In [1]:
!pip install opendatasets

^C


Defaulting to user installation because normal site-packages is not writeable



[notice] A new release of pip is available: 25.0.1 -> 26.1.2
[notice] To update, run: C:\Users\divak\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [ ]:
import opendatasets as od
od.download("https://www.kaggle.com/datasets/orvile/wesad-wearable-stress-affect-detection-dataset")

In [ ]:
import numpy as np
import pandas as pd
import pickle
import json
import warnings
import time
import os
from pathlib import Path
from collections import defaultdict

import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns

from scipy import signal, stats
from scipy.fft import fft, fftfreq

from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score, f1_score, confusion_matrix,
    classification_report, roc_auc_score
)
from sklearn.base import clone

warnings.filterwarnings("ignore")
np.random.seed(42)

In [ ]:
# ── Configuration ──────────────────────────────────────────────────
USE_REAL_DATA    = True
WESAD_ROOT       = "/content/wesad-wearable-stress-affect-detection-dataset/WESAD"

WESAD_PKL_PATHS = [
    "/content/wesad-wearable-stress-affect-detection-dataset/WESAD/S2/S2.pkl",
    "/content/wesad-wearable-stress-affect-detection-dataset/WESAD/S3/S3.pkl",
    "/content/wesad-wearable-stress-affect-detection-dataset/WESAD/S4/S4.pkl",
    "/content/wesad-wearable-stress-affect-detection-dataset/WESAD/S5/S5.pkl",
    "/content/wesad-wearable-stress-affect-detection-dataset/WESAD/S6/S6.pkl",
    "/content/wesad-wearable-stress-affect-detection-dataset/WESAD/S7/S7.pkl",
    "/content/wesad-wearable-stress-affect-detection-dataset/WESAD/S8/S8.pkl",
    "/content/wesad-wearable-stress-affect-detection-dataset/WESAD/S9/S9.pkl",
    "/content/wesad-wearable-stress-affect-detection-dataset/WESAD/S10/S10.pkl",
    "/content/wesad-wearable-stress-affect-detection-dataset/WESAD/S11/S11.pkl",
    "/content/wesad-wearable-stress-affect-detection-dataset/WESAD/S13/S13.pkl",
    "/content/wesad-wearable-stress-affect-detection-dataset/WESAD/S14/S14.pkl",
    "/content/wesad-wearable-stress-affect-detection-dataset/WESAD/S15/S15.pkl",
    "/content/wesad-wearable-stress-affect-detection-dataset/WESAD/S16/S16.pkl",
    "/content/wesad-wearable-stress-affect-detection-dataset/WESAD/S17/S17.pkl",
]

SAMPLE_RATE   = 700   # Hz (chest RespiBAN)
WINDOW_SEC    = 60    # seconds per feature window
STEP_SEC      = 30    # sliding step in seconds
N_SUBJECTS    = 15    # used only when USE_REAL_DATA = False (synthetic mode)

LABEL_MAP  = {1: "Baseline", 2: "Stress", 3: "Amusement", 4: "Meditation"}
LABEL_COLS = {1: "#4C72B0", 2: "#C44E52", 3: "#55A868", 4: "#DD8452"}

print("✅ Imports complete")
print(f"   Mode      : {'Real WESAD dataset' if USE_REAL_DATA else 'Synthetic data'}")
print(f"   SR        : {SAMPLE_RATE} Hz")
print(f"   Window    : {WINDOW_SEC}s  step {STEP_SEC}s")
if USE_REAL_DATA:
    print(f"   Subjects  : {len(WESAD_PKL_PATHS)} pkl files configured")
    print(f"   Dataset   : {WESAD_ROOT}")
else:
    print(f"   Subjects  : {N_SUBJECTS} (synthetic)")



In [ ]:
# CELL 3 — Synthetic WESAD data generator
# ──────────────────────────────────────────────────────────────────

def _band_power(sig, sr, lo, hi):
    nperseg = min(len(sig), 512)
    freqs, psd = signal.welch(sig, fs=sr, nperseg=nperseg)
    idx = (freqs >= lo) & (freqs <= hi)
    return float(np.trapezoid(psd[idx], freqs[idx])) if idx.sum() > 0 else 0.0


def simulate_ecg(n_samples, sr, condition):
    """ECG with condition-specific heart rate and HRV."""
    cfg = dict(
        baseline  = dict(hr=70,  hrv=0.06, noise=0.05),
        stress    = dict(hr=90,  hrv=0.02, noise=0.09),
        amusement = dict(hr=75,  hrv=0.06, noise=0.06),
        meditation= dict(hr=58,  hrv=0.09, noise=0.04),
    )[condition]
    dur = n_samples / sr
    hr  = cfg["hr"] + np.random.randn() * 4
    rr  = 60.0 / hr
    ecg = np.zeros(n_samples)
    beats = np.arange(0, dur, rr)
    beats += np.cumsum(np.random.randn(len(beats)) * cfg["hrv"] * rr)
    for bt in beats:
        idx = int(bt * sr)
        if idx >= n_samples:
            break
        for di in range(-int(0.05 * sr), int(0.05 * sr)):
            if 0 <= idx + di < n_samples:
                ecg[idx + di] += np.exp(-0.5 * (di / (0.02 * sr)) ** 2)
    ecg += np.random.randn(n_samples) * cfg["noise"]
    return ecg


def simulate_eda(n_samples, sr, condition):
    """EDA with tonic level and phasic SCR bursts."""
    cfg = dict(
        baseline  = dict(base=2.0, rate=0.4, amp=0.3),
        stress    = dict(base=5.5, rate=2.2, amp=1.6),
        amusement = dict(base=3.2, rate=1.0, amp=0.9),
        meditation= dict(base=1.4, rate=0.15,amp=0.2),
    )[condition]
    eda = np.ones(n_samples) * cfg["base"] + np.random.randn(n_samples) * 0.08
    dur = n_samples / sr
    for _ in range(int(cfg["rate"] * dur)):
        onset = np.random.randint(0, max(1, n_samples - int(sr * 4)))
        amp   = abs(np.random.randn()) * cfg["amp"]
        for di in range(min(int(sr * 4), n_samples - onset)):
            t = di / sr
            eda[onset + di] += amp * t * np.exp(-t / 0.9)
    return eda


def simulate_resp(n_samples, sr, condition):
    """Respiration signal with condition-specific breathing rate."""
    cfg = dict(
        baseline  = dict(rate=15, amp=1.0),
        stress    = dict(rate=21, amp=0.75),
        amusement = dict(rate=17, amp=1.1),
        meditation= dict(rate=9,  amp=1.4),
    )[condition]
    t    = np.linspace(0, n_samples / sr, n_samples)
    freq = cfg["rate"] / 60.0
    resp = cfg["amp"] * np.sin(2 * np.pi * freq * t + np.random.uniform(0, 2 * np.pi))
    resp += np.random.randn(n_samples) * 0.08
    return resp


def simulate_temp(n_samples, sr, condition):
    """Skin temperature with slow drift."""
    cfg = dict(
        baseline  = dict(mean=33.0, std=0.08),
        stress    = dict(mean=31.8, std=0.28),
        amusement = dict(mean=33.6, std=0.12),
        meditation= dict(mean=34.1, std=0.06),
    )[condition]
    return np.cumsum(np.random.randn(n_samples) * 0.001) + cfg["mean"] \
           + np.random.randn(n_samples) * cfg["std"] * 0.1


def simulate_emg(n_samples, condition):
    """EMG amplitude correlates with muscle tension / arousal."""
    amp = dict(baseline=0.10, stress=0.42, amusement=0.18, meditation=0.04)[condition]
    return np.random.randn(n_samples) * amp


def generate_subject(subject_id):
    """Generate one subject's full session."""
    protocol = [(1,60),(2,90),(3,60),(4,30),(2,90),(3,60)]
    signals  = {k: [] for k in ["ecg","eda","resp","temp","emg"]}
    labels   = []
    for lbl, dur in protocol:
        cond = LABEL_MAP[lbl].lower()
        n    = dur * SAMPLE_RATE
        signals["ecg" ].append(simulate_ecg (n, SAMPLE_RATE, cond))
        signals["eda" ].append(simulate_eda (n, SAMPLE_RATE, cond))
        signals["resp"].append(simulate_resp(n, SAMPLE_RATE, cond))
        signals["temp"].append(simulate_temp(n, SAMPLE_RATE, cond))
        signals["emg" ].append(simulate_emg (n, cond))
        labels.extend([lbl] * n)
    df = pd.DataFrame({k: np.concatenate(v) for k, v in signals.items()})
    df["label"]   = labels
    df["subject"] = subject_id
    return df


def generate_synthetic_dataset(n_subjects=N_SUBJECTS):
    print(f"Generating synthetic WESAD dataset ({n_subjects} subjects)…")
    dfs = []
    for s in range(1, n_subjects + 1):
        df = generate_subject(s)
        dfs.append(df)
        print(f"  S{s:02d} ✓  {len(df):,} samples", end="\r")
    full = pd.concat(dfs, ignore_index=True)
    print(f"\n✅ Total: {len(full):,} samples  |  "
          f"Subjects: {full['subject'].nunique()}  |  "
          f"Signals: ECG, EDA, RESP, TEMP, EMG")
    return full



In [ ]:
# CELL 4 — Real WESAD loader (used only when USE_REAL_DATA = True)
# ──────────────────────────────────────────────────────────────────

def load_real_wesad(pkl_paths=None):
    """
    Load real WESAD .pkl files directly from /content/ (Kaggle dataset mount).
    Uses the hardcoded WESAD_PKL_PATHS list — no Google Drive mount required.

    Each .pkl file has structure:
        data['signal']['chest'] → dict with keys: ECG, EDA, EMG, Resp, Temp, ACC
        data['label']           → 1-D array aligned sample-by-sample with signals
    Labels: 0=transient (excluded), 1=Baseline, 2=Stress, 3=Amusement, 4=Meditation
    """
    if pkl_paths is None:
        pkl_paths = WESAD_PKL_PATHS

    print(f"Loading {len(pkl_paths)} WESAD subject files from /content/...")
    dfs = []
    for pkl_path in pkl_paths:
        pkl_path = Path(pkl_path)
        sid = int(pkl_path.stem[1:])

        if not pkl_path.exists():
            print(f"  ⚠  S{sid:02d} — file not found: {pkl_path}")
            continue

        try:
            with open(pkl_path, "rb") as f:
                data = pickle.load(f, encoding="latin1")

            chest = data["signal"]["chest"]
            lbl   = data["label"].flatten().astype(int)
            n     = min(len(lbl), chest["ECG"].shape[0])

            df = pd.DataFrame({
                "ecg":     chest["ECG"] [:n].flatten(),
                "eda":     chest["EDA"] [:n].flatten(),
                "emg":     chest["EMG"] [:n].flatten(),
                "resp":    chest["Resp"][:n].flatten(),
                "temp":    chest["Temp"][:n].flatten(),
                "label":   lbl[:n],
                "subject": sid,
            })

            df = df[df["label"].isin([1, 2, 3, 4])].reset_index(drop=True)

            label_counts = df["label"].value_counts().sort_index()
            label_str = "  ".join(
                f"{LABEL_MAP[l]}:{cnt:,}" for l, cnt in label_counts.items()
            )
            print(f"  S{sid:02d} ✓  {len(df):,} labeled samples  [{label_str}]")
            dfs.append(df)

        except Exception as e:
            print(f"  ✗  S{sid:02d} — failed to load: {e}")
            continue

    if not dfs:
        raise RuntimeError(
            "No subject files loaded. Check that the dataset paths in "
            "WESAD_PKL_PATHS are correct and the Kaggle dataset is attached."
        )

    full = pd.concat(dfs, ignore_index=True)
    print(f"\n✅ Real WESAD loaded:")
    print(f"   Total samples : {len(full):,}")
    print(f"   Subjects      : {full['subject'].nunique()}  "
          f"({sorted(full['subject'].unique())})")
    print(f"   Label distribution:")
    for lbl, name in LABEL_MAP.items():
        n = (full["label"] == lbl).sum()
        pct = 100 * n / len(full)
        print(f"     {name:12s}: {n:>8,}  ({pct:.1f}%)")
    return full



In [ ]:
# CELL 5 — Feature extraction
# ──────────────────────────────────────────────────────────────────

def detect_r_peaks(ecg, sr):
    min_dist = int(0.3 * sr)
    threshold = np.mean(ecg) + 0.3 * np.std(ecg)
    peaks, _ = signal.find_peaks(ecg, distance=min_dist, height=threshold)
    return peaks


def ecg_features(ecg, sr):
    peaks = detect_r_peaks(ecg, sr)
    if len(peaks) < 4:
        return {f"ecg_{k}": 0.0 for k in
                ["hr_mean","hr_std","rmssd","sdnn","pnn50",
                 "lf_power","hf_power","lf_hf","sd1","sd2","sampen"]}
    rr = np.diff(peaks) / sr * 1000  # ms
    hr = 60000 / rr
    feats = {
        "ecg_hr_mean": float(np.mean(hr)),
        "ecg_hr_std":  float(np.std(hr)),
        "ecg_rmssd":   float(np.sqrt(np.mean(np.diff(rr)**2))),
        "ecg_sdnn":    float(np.std(rr)),
        "ecg_pnn50":   float(100 * np.sum(np.abs(np.diff(rr)) > 50) / len(rr)),
    }
    if len(rr) > 5:
        rr_t   = np.cumsum(rr) / 1000
        t_i    = np.arange(rr_t[0], rr_t[-1], 0.25)
        rr_i   = np.interp(t_i, rr_t, rr)
        feats["ecg_lf_power"] = _band_power(rr_i, 4, 0.04, 0.15)
        feats["ecg_hf_power"] = _band_power(rr_i, 4, 0.15, 0.40)
        feats["ecg_lf_hf"]    = feats["ecg_lf_power"] / (feats["ecg_hf_power"] + 1e-9)
    else:
        feats.update({"ecg_lf_power":0.0,"ecg_hf_power":0.0,"ecg_lf_hf":0.0})
    sd1 = float(np.std(np.diff(rr)) / np.sqrt(2))
    sd2 = float(np.sqrt(max(0, 2*np.var(rr) - 0.5*np.var(np.diff(rr)))))
    feats["ecg_sd1"]   = sd1
    feats["ecg_sd2"]   = sd2
    feats["ecg_sampen"]= float(np.std(rr) / (np.mean(rr) + 1e-9))
    return feats


def eda_features(eda, sr):
    b, a  = signal.butter(3, 0.05 / (sr/2), btype="low")
    scl   = signal.filtfilt(b, a, eda)
    phasic = eda - scl
    peaks, props = signal.find_peaks(phasic, height=0.01, distance=int(sr))
    return {
        "eda_scl_mean":      float(np.mean(scl)),
        "eda_scl_std":       float(np.std(scl)),
        "eda_scl_range":     float(np.ptp(scl)),
        "eda_scr_mean":      float(np.mean(phasic)),
        "eda_scr_std":       float(np.std(phasic)),
        "eda_n_peaks":       float(len(peaks)),
        "eda_peak_amp_mean": float(np.mean(props["peak_heights"])) if len(peaks) > 0 else 0.0,
        "eda_skewness":      float(stats.skew(eda)),
        "eda_kurtosis":      float(stats.kurtosis(eda)),
    }


def resp_features(resp, sr):
    freqs, psd = signal.welch(resp, fs=sr, nperseg=min(len(resp), 2048))
    band = (freqs >= 0.1) & (freqs <= 0.5)
    rate = (freqs[band][np.argmax(psd[band])] * 60) if band.sum() > 0 else 15.0
    pos  = resp - resp.mean()
    return {
        "resp_rate":     float(rate),
        "resp_amp_mean": float(np.mean(np.abs(pos))),
        "resp_amp_std":  float(np.std(resp)),
        "resp_power":    _band_power(resp, sr, 0.1, 0.5),
        "resp_ie_ratio": float(np.sum(pos > 0) / (np.sum(pos < 0) + 1)),
    }


def temp_features(temp):
    return {
        "temp_mean":  float(np.mean(temp)),
        "temp_std":   float(np.std(temp)),
        "temp_min":   float(np.min(temp)),
        "temp_max":   float(np.max(temp)),
        "temp_slope": float(np.polyfit(np.arange(len(temp)), temp, 1)[0]),
    }


def emg_features(emg, sr):
    env = np.abs(signal.hilbert(emg))
    return {
        "emg_rms":      float(np.sqrt(np.mean(emg**2))),
        "emg_mean_abs": float(np.mean(np.abs(emg))),
        "emg_std":      float(np.std(emg)),
        "emg_power":    _band_power(emg, sr, 20, 150),
        "emg_env_mean": float(np.mean(env)),
    }


def extract_window(win, sr=SAMPLE_RATE):
    feats = {}
    feats.update(ecg_features (win["ecg" ].values, sr))
    feats.update(eda_features (win["eda" ].values, sr))
    feats.update(resp_features(win["resp"].values, sr))
    feats.update(temp_features(win["temp"].values))
    feats.update(emg_features (win["emg" ].values, sr))
    return feats


def build_feature_matrix(df, sr=SAMPLE_RATE, window_sec=WINDOW_SEC, step_sec=STEP_SEC):
    ws   = window_sec * sr
    step = step_sec   * sr
    rows = []
    subjects = sorted(df["subject"].unique())
    total_wins = 0

    print(f"Extracting features  (window={window_sec}s, step={step_sec}s)…")
    for subj in subjects:
        sdf = df[df["subject"] == subj].reset_index(drop=True)
        n   = len(sdf)
        wins = 0
        for start in range(0, n - ws + 1, step):
            win   = sdf.iloc[start : start + ws]
            label = int(win["label"].mode()[0])
            if label not in LABEL_MAP:
                continue
            feats           = extract_window(win, sr)
            feats["label"]  = label
            feats["subject"]= int(subj)
            rows.append(feats)
            wins += 1
        total_wins += wins

    feat_df = pd.DataFrame(rows)
    feat_cols = [c for c in feat_df.columns if c not in ("label","subject")]
    print(f"✅ Feature matrix: {feat_df.shape[0]} windows × {len(feat_cols)} features")
    print(f"   Label distribution:")
    for lbl, name in LABEL_MAP.items():
        n = (feat_df["label"] == lbl).sum()
        print(f"     {name:12s}: {n}")
    return feat_df


In [ ]:
# CELL 6 — LOSO cross-validation
# ──────────────────────────────────────────────────────────────────

CLASSIFIERS = {
    "Random Forest":      RandomForestClassifier(n_estimators=200, max_depth=15,
                                                  n_jobs=-1, random_state=42),
    "Gradient Boosting":  GradientBoostingClassifier(n_estimators=100,
                                                      learning_rate=0.05,
                                                      max_depth=5, random_state=42),
    "SVM (RBF)":          SVC(C=1.0, kernel="rbf", probability=True, random_state=42),
    "Logistic Regression":LogisticRegression(C=0.1, max_iter=1000, random_state=42),
}


def loso_cv(feat_df, binary=True):
    """Leave-One-Subject-Out cross-validation."""
    feat_cols = [c for c in feat_df.columns if c not in ("label","subject")]
    subjects  = sorted(feat_df["subject"].unique())
    results   = {}

    task = "Binary (Stress vs. Rest)" if binary else "Multi-class (4 conditions)"
    print(f"\n{'─'*55}")
    print(f"  {task}")
    print(f"{'─'*55}")

    for name, base_clf in CLASSIFIERS.items():
        all_true, all_pred = [], []
        subj_res = []

        for test_subj in subjects:
            tr = feat_df[feat_df["subject"] != test_subj]
            te = feat_df[feat_df["subject"] == test_subj]

            y_tr = (tr["label"] == 2).astype(int) if binary else tr["label"].values
            y_te = (te["label"] == 2).astype(int) if binary else te["label"].values

            scaler = StandardScaler()
            X_tr   = scaler.fit_transform(tr[feat_cols].fillna(0))
            X_te   = scaler.transform   (te[feat_cols].fillna(0))

            clf = clone(base_clf)
            clf.fit(X_tr, y_tr)
            y_pred = clf.predict(X_te)

            subj_res.append({
                "subject":  test_subj,
                "accuracy": accuracy_score(y_te, y_pred),
                "f1":       f1_score(y_te, y_pred, average="weighted", zero_division=0),
            })
            all_true.extend(y_te.tolist())
            all_pred.extend(y_pred.tolist())

        oa  = accuracy_score(all_true, all_pred)
        of1 = f1_score(all_true, all_pred, average="weighted", zero_division=0)
        cm  = confusion_matrix(all_true, all_pred)
        results[name] = dict(accuracy=oa, f1=of1, cm=cm,
                              subj_results=subj_res,
                              y_true=all_true, y_pred=all_pred)
        print(f"  {name:<25}  acc={oa:.3f}  f1={of1:.3f}")

    return results


def run_all_loso(feat_df):
    print("\n" + "═"*55)
    print("  LEAVE-ONE-SUBJECT-OUT CROSS-VALIDATION")
    print("═"*55)
    binary_res = loso_cv(feat_df, binary=True)
    multi_res  = loso_cv(feat_df, binary=False)
    return binary_res, multi_res



In [ ]:
# CELL 7 — Feature importance
# ──────────────────────────────────────────────────────────────────

def compute_feature_importance(feat_df):
    feat_cols = [c for c in feat_df.columns if c not in ("label","subject")]
    X = StandardScaler().fit_transform(feat_df[feat_cols].fillna(0))
    y = (feat_df["label"] == 2).astype(int)
    rf = RandomForestClassifier(n_estimators=300, random_state=42, n_jobs=-1)
    rf.fit(X, y)
    fi = pd.Series(rf.feature_importances_, index=feat_cols).sort_values(ascending=False)
    print(f"✅ Feature importance computed (top signal: {fi.index[0]})")
    return fi


In [ ]:
# CELL 8 — Train & save final production model
# ──────────────────────────────────────────────────────────────────

def train_final_model(feat_df):
    feat_cols = [c for c in feat_df.columns if c not in ("label","subject")]
    scaler    = StandardScaler()
    X         = scaler.fit_transform(feat_df[feat_cols].fillna(0))
    y_bin     = (feat_df["label"] == 2).astype(int)
    y_multi   = feat_df["label"].values

    rf_bin   = RandomForestClassifier(n_estimators=300, random_state=42, n_jobs=-1)
    rf_multi = RandomForestClassifier(n_estimators=300, random_state=42, n_jobs=-1)
    rf_bin  .fit(X, y_bin)
    rf_multi.fit(X, y_multi)

    payload = {
        "binary_model":  rf_bin,
        "multi_model":   rf_multi,
        "scaler":        scaler,
        "feature_cols":  feat_cols,
        "label_map":     LABEL_MAP,
    }
    out = "/tmp/wesad_stress_model.pkl" if not os.path.isdir("/content") else "/content/wesad_stress_model.pkl"
    with open(out, "wb") as f:
        pickle.dump(payload, f)
    print(f"✅ Final model saved → {out}")
    return payload



In [ ]:
# CELL 9 — Inference helper (use the saved model on new windows)
# ──────────────────────────────────────────────────────────────────

def predict_stress(model_payload, raw_window_df):
    """
    Predict stress from a raw 60-second window DataFrame.

    Parameters
    ----------
    model_payload : dict  – loaded from wesad_stress_model.pkl
    raw_window_df : DataFrame with columns [ecg, eda, resp, temp, emg]
                    containing exactly 60 * 700 = 42,000 rows

    Returns
    -------
    dict with keys:
        stress_prob  – probability of stress (0–1)
        binary_label – 0=Not stressed  1=Stressed
        multi_label  – one of Baseline/Stress/Amusement/Meditation
    """
    feats   = extract_window(raw_window_df)
    X       = pd.DataFrame([feats])[model_payload["feature_cols"]].fillna(0)
    X_sc    = model_payload["scaler"].transform(X)
    bin_prob  = model_payload["binary_model"].predict_proba(X_sc)[0][1]
    bin_label = int(bin_prob > 0.5)
    multi_lbl = int(model_payload["multi_model"].predict(X_sc)[0])
    return {
        "stress_prob":  round(float(bin_prob), 4),
        "binary_label": bin_label,
        "binary_text":  "Stressed" if bin_label else "Not stressed",
        "multi_label":  multi_lbl,
        "multi_text":   LABEL_MAP[multi_lbl],
    }


In [ ]:
# CELL 10 — Visualisation
# ──────────────────────────────────────────────────────────────────

def plot_all(feat_df, binary_res, multi_res, feature_importance):
    sns.set_theme(style="whitegrid", font_scale=1.0)
    fig = plt.figure(figsize=(20, 22))
    fig.patch.set_facecolor("#F8F8F8")
    gs  = gridspec.GridSpec(4, 3, figure=fig, hspace=0.52, wspace=0.38)

    models   = list(CLASSIFIERS.keys())
    short_m  = ["RF", "GBM", "SVM", "LR"]
    bin_acc  = [binary_res[m]["accuracy"] for m in models]
    bin_f1   = [binary_res[m]["f1"]       for m in models]
    multi_acc= [multi_res [m]["accuracy"] for m in models]
    multi_f1 = [multi_res [m]["f1"]       for m in models]

    # ── 1. Binary accuracy ──────────────────────────────────────────
    ax1 = fig.add_subplot(gs[0, 0])
    x   = np.arange(len(models))
    w   = 0.35
    bars1 = ax1.bar(x - w/2, bin_acc, w, label="Accuracy", color="#4C72B0", alpha=0.85)
    bars2 = ax1.bar(x + w/2, bin_f1,  w, label="F1 (weighted)", color="#55A868", alpha=0.85)
    ax1.set_xticks(x); ax1.set_xticklabels(short_m)
    ax1.set_ylim(0.75, 1.05); ax1.set_title("Binary classification\n(Stress vs. Rest)", fontweight="bold")
    ax1.set_ylabel("Score"); ax1.legend(fontsize=8)
    for bar in list(bars1)+list(bars2):
        ax1.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.003,
                 f"{bar.get_height():.3f}", ha="center", va="bottom", fontsize=7)

    # ── 2. Multi-class accuracy ─────────────────────────────────────
    ax2 = fig.add_subplot(gs[0, 1])
    bars3 = ax2.bar(x - w/2, multi_acc, w, color="#C44E52", alpha=0.85, label="Accuracy")
    bars4 = ax2.bar(x + w/2, multi_f1,  w, color="#DD8452", alpha=0.85, label="F1 (weighted)")
    ax2.set_xticks(x); ax2.set_xticklabels(short_m)
    ax2.set_ylim(0.75, 1.05); ax2.set_title("Multi-class classification\n(4 conditions)", fontweight="bold")
    ax2.set_ylabel("Score"); ax2.legend(fontsize=8)
    for bar in list(bars3)+list(bars4):
        ax2.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.003,
                 f"{bar.get_height():.3f}", ha="center", va="bottom", fontsize=7)

    # ── 3. Label distribution ───────────────────────────────────────
    ax3  = fig.add_subplot(gs[0, 2])
    dist = feat_df["label"].value_counts().sort_index()
    wedge_cols = [LABEL_COLS[i] for i in dist.index]
    wedges, texts, autotexts = ax3.pie(
        dist.values, labels=[LABEL_MAP[i] for i in dist.index],
        autopct="%1.1f%%", colors=wedge_cols, startangle=140,
        textprops={"fontsize": 9})
    for at in autotexts: at.set_fontsize(8)
    ax3.set_title("Window label distribution", fontweight="bold")

    # ── 4. Confusion matrix — RF binary ────────────────────────────
    ax4 = fig.add_subplot(gs[1, 0])
    cm  = np.array(binary_res["Random Forest"]["cm"])
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", ax=ax4, linewidths=0.5,
                xticklabels=["Non-stress","Stress"],
                yticklabels=["Non-stress","Stress"])
    ax4.set_title("Confusion matrix\n(RF binary, LOSO-CV)", fontweight="bold")
    ax4.set_xlabel("Predicted"); ax4.set_ylabel("True")

    # ── 5. Confusion matrix — RF multi ─────────────────────────────
    ax5 = fig.add_subplot(gs[1, 1])
    cm5 = np.array(multi_res["Random Forest"]["cm"])
    existing_labels = sorted(feat_df["label"].unique())
    label_names     = [LABEL_MAP[l] for l in existing_labels]
    sns.heatmap(cm5, annot=True, fmt="d", cmap="Reds", ax=ax5,
                xticklabels=label_names, yticklabels=label_names,
                linewidths=0.5)
    ax5.set_title("Confusion matrix\n(RF multi-class, LOSO-CV)", fontweight="bold")
    ax5.set_xticklabels(label_names, rotation=30, ha="right", fontsize=8)
    ax5.set_yticklabels(label_names, rotation=0, fontsize=8)
    ax5.set_xlabel("Predicted"); ax5.set_ylabel("True")

    # ── 6. Per-subject accuracy ─────────────────────────────────────
    ax6 = fig.add_subplot(gs[1, 2])
    sr_  = binary_res["Random Forest"]["subj_results"]
    subj_ids  = [r["subject"]  for r in sr_]
    subj_accs = [r["accuracy"] for r in sr_]
    colors_s  = ["#55A868" if a >= 0.99 else "#DD8452" if a >= 0.90 else "#C44E52"
                 for a in subj_accs]
    ax6.bar([f"S{s:02d}" for s in subj_ids], subj_accs, color=colors_s, alpha=0.85)
    ax6.set_ylim(0.7, 1.05); ax6.set_title("Per-subject accuracy\n(RF binary, LOSO)", fontweight="bold")
    ax6.set_ylabel("Accuracy"); ax6.tick_params(axis="x", rotation=45, labelsize=7)
    ax6.axhline(np.mean(subj_accs), ls="--", color="#4C72B0", lw=1.2,
                label=f"Mean={np.mean(subj_accs):.3f}")
    ax6.legend(fontsize=8)

    # ── 7. Top 15 feature importances ──────────────────────────────
    ax7  = fig.add_subplot(gs[2, :])
    top  = feature_importance.head(15)
    sig_colors = {"ecg":"#4C72B0","eda":"#55A868","resp":"#DD8452",
                  "temp":"#C44E52","emg":"#9467BD"}
    bar_cols = [sig_colors.get(n.split("_")[0], "#888") for n in top.index]
    ax7.barh(range(len(top)), top.values[::-1], color=bar_cols[::-1], alpha=0.85)
    ax7.set_yticks(range(len(top)))
    ax7.set_yticklabels(top.index[::-1], fontsize=9)
    ax7.set_xlabel("Importance")
    ax7.set_title("Top 15 feature importances (Random Forest, binary stress detection)", fontweight="bold")
    handles = [plt.Rectangle((0,0),1,1, color=c) for c in sig_colors.values()]
    ax7.legend(handles, sig_colors.keys(), loc="lower right", fontsize=8)

    # ── 8. Simulated ECG per condition ─────────────────────────────
    ax8 = fig.add_subplot(gs[3, 0])
    sr_plot = 700; dur_plot = 3
    for lbl, name in LABEL_MAP.items():
        cond = name.lower()
        ecg_s = simulate_ecg(dur_plot * sr_plot, sr_plot, cond)
        t = np.linspace(0, dur_plot, len(ecg_s))
        ax8.plot(t, ecg_s + (lbl-1)*3, color=LABEL_COLS[lbl], lw=0.8, label=name)
    ax8.set_title("Simulated ECG\n(3s per condition, offset for clarity)", fontweight="bold")
    ax8.set_xlabel("Time (s)"); ax8.legend(fontsize=7)
    ax8.set_yticks([])

    # ── 9. EDA per condition ────────────────────────────────────────
    ax9 = fig.add_subplot(gs[3, 1])
    for lbl, name in LABEL_MAP.items():
        cond = name.lower()
        n_pts = 5 * sr_plot
        eda_s = simulate_eda(n_pts, sr_plot, cond)
        t = np.linspace(0, 5, n_pts)
        ax9.plot(t, eda_s, color=LABEL_COLS[lbl], lw=1.0, label=name)
    ax9.set_title("Simulated EDA\n(5s per condition)", fontweight="bold")
    ax9.set_xlabel("Time (s)"); ax9.set_ylabel("μS")
    ax9.legend(fontsize=7)

    # ── 10. Radar — physiological profile per condition ─────────────
    ax10 = fig.add_subplot(gs[3, 2], polar=True)
    radar_labels = ["HR\n(↑ stress)", "EDA\n(↑ stress)", "EMG\n(↑ stress)",
                    "Resp\nrate", "HRV\n(↓ stress)", "Temp\n(↓ stress)"]
    radar_data = {
        "Baseline":   [60, 40, 18, 50, 70, 60],
        "Stress":     [92, 88, 80, 76, 28, 30],
        "Amusement":  [66, 58, 32, 56, 62, 56],
        "Meditation": [46, 22,  8, 34, 88, 72],
    }
    N_ax  = len(radar_labels)
    angles = [n / N_ax * 2 * np.pi for n in range(N_ax)]
    angles += angles[:1]
    ax10.set_theta_offset(np.pi / 2); ax10.set_theta_direction(-1)
    ax10.set_xticks(angles[:-1]); ax10.set_xticklabels(radar_labels, size=7)
    ax10.set_yticks([]); ax10.set_ylim(0, 100)
    for lbl, name in LABEL_MAP.items():
        vals = radar_data[name] + radar_data[name][:1]
        ax10.plot(angles, vals, color=LABEL_COLS[lbl], lw=1.5, label=name)
        ax10.fill(angles, vals, color=LABEL_COLS[lbl], alpha=0.06)
    ax10.set_title("Physiological\nprofile by condition", fontweight="bold", pad=15)
    ax10.legend(loc="upper right", bbox_to_anchor=(1.35, 1.1), fontsize=7)

    fig.suptitle("WESAD Stress Detection — Full Pipeline Results",
                 fontsize=16, fontweight="bold", y=1.01)
    plt.savefig(("/tmp" if not os.path.isdir("/content") else "/content") + "/wesad_results.png", dpi=150, bbox_inches="tight",
                facecolor=fig.get_facecolor())
    plt.show()
    print("✅ Plot saved → /content/wesad_results.png")



In [ ]:
# CELL 11 — Print leaderboard summary
# ──────────────────────────────────────────────────────────────────

def print_leaderboard(binary_res, multi_res):
    print("\n" + "═"*68)
    print("  LEADERBOARD  (LOSO cross-validation)")
    print("═"*68)
    print(f"  {'Model':<25} {'Bin Acc':>9} {'Bin F1':>8} {'Multi Acc':>10} {'Multi F1':>9}")
    print("  " + "─"*64)
    for name in CLASSIFIERS:
        b = binary_res[name];  m = multi_res[name]
        best_b = "⭐" if b["accuracy"] == max(binary_res[n]["accuracy"] for n in CLASSIFIERS) else "  "
        print(f"  {best_b}{name:<23} {b['accuracy']:>9.3f} {b['f1']:>8.3f} "
              f"{m['accuracy']:>10.3f} {m['f1']:>9.3f}")
    print("═"*68)

    print("\n  TOP FEATURES (stress detection):")
    print("  " + "─"*40)


def print_feature_table(fi, top_n=10):
    sig_map = {"ecg":"ECG","eda":"EDA","resp":"RESP","temp":"TEMP","emg":"EMG"}
    for feat, imp in fi.head(top_n).items():
        sig = sig_map.get(feat.split("_")[0], "?")
        bar = "█" * int(imp / fi.iloc[0] * 20)
        print(f"  {feat:<32} [{sig:4s}]  {bar:<20}  {imp:.4f}")



In [ ]:
# CELL 12 — MAIN  (run everything)
# ──────────────────────────────────────────────────────────────────

def main():
    t0 = time.time()
    print("╔══════════════════════════════════════════════════════╗")
    print("║       WESAD STRESS DETECTION · COLAB PIPELINE       ║")
    print("╚══════════════════════════════════════════════════════╝\n")

    # ── Step 1: Load / generate data ──────────────────────────────
    print("STEP 1 — Data")
    if USE_REAL_DATA:
        raw_df = load_real_wesad()          # uses WESAD_PKL_PATHS defined above
    else:
        raw_df = generate_synthetic_dataset(N_SUBJECTS)

    # ── Step 2: Feature extraction ─────────────────────────────────
    print("\nSTEP 2 — Feature extraction")
    feat_df = build_feature_matrix(raw_df)

    # ── Step 3: LOSO cross-validation ─────────────────────────────
    print("\nSTEP 3 — Model training & evaluation (LOSO-CV)")
    binary_res, multi_res = run_all_loso(feat_df)

    # ── Step 4: Feature importance ─────────────────────────────────
    print("\nSTEP 4 — Feature importance")
    fi = compute_feature_importance(feat_df)

    # ── Step 5: Train final model ──────────────────────────────────
    print("\nSTEP 5 — Final model training")
    model_payload = train_final_model(feat_df)

    # ── Step 6: Visualise ──────────────────────────────────────────
    print("\nSTEP 6 — Visualisation")
    plot_all(feat_df, binary_res, multi_res, fi)

    # ── Leaderboard ────────────────────────────────────────────────
    print_leaderboard(binary_res, multi_res)
    print_feature_table(fi)

    # ── Quick inference demo ───────────────────────────────────────
    print("\n  INFERENCE DEMO (simulated stress window):")
    print("  " + "─"*44)
    n_demo   = WINDOW_SEC * SAMPLE_RATE
    demo_win = pd.DataFrame({
        "ecg":  simulate_ecg (n_demo, SAMPLE_RATE, "stress"),
        "eda":  simulate_eda (n_demo, SAMPLE_RATE, "stress"),
        "resp": simulate_resp(n_demo, SAMPLE_RATE, "stress"),
        "temp": simulate_temp(n_demo, SAMPLE_RATE, "stress"),
        "emg":  simulate_emg (n_demo, "stress"),
    })
    pred = predict_stress(model_payload, demo_win)
    print(f"  Input      : Simulated stress window ({WINDOW_SEC}s)")
    print(f"  Stress prob: {pred['stress_prob']:.4f}")
    print(f"  Binary     : {pred['binary_text']}")
    print(f"  Multi-class: {pred['multi_text']}")

    print(f"\n⏱  Total time: {time.time()-t0:.1f}s")
    print("\n📁 Output files:")
    print("   /content/wesad_stress_model.pkl (or /tmp/ locally)  — trained model + scaler")
    print("   /content/wesad_results.png (or /tmp/ locally)        — full results figure")
    print("\n🎉 Done!\n")


# ── Entry point ────────────────────────────────────────────────────
if __name__ == "__main__":
    main()
